# Travel Assistant - Optimization Insights (Fabric reverse-ETL)

Reads the **mirrored** Cosmos analytics tables, computes optimization KPIs +
per-tier costs entirely in Spark, and writes them back to Cosmos
(`OptimizationInsights`) via the **Cosmos Spark connector** using the Fabric
identity (workspace identity has Cosmos **Data Contributor**). No keys, no pip
(so it runs in scheduled jobs, not just interactively).

In [ ]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.azure.cosmos.spark:azure-cosmos-spark_3-5_2-12:4.41.0,com.azure.cosmos.spark:fabric-cosmos-spark-auth_3:1.1.0"
    }
}

In [ ]:

from pyspark.sql import functions as F
from datetime import datetime, timezone

# ---- config (workshop dev) ----
WORKSPACE_ID = "37733bf9-e6c2-4472-b4fb-22cb547079f7"
MIRROR_ID    = "debe9a19-56de-4363-9ea9-88dc29baa003"
COSMOS_ENDPOINT = "https://cosmos-kfpokdh52vbec.documents.azure.com:443/"
COSMOS_DATABASE = "TravelAssistantV2"
INSIGHTS_CONTAINER = "OptimizationInsights"
TENANT_ID = "72f988bf-86f1-41af-91ab-2d7cd011db47"
MIRROR_NAME = "TravelAssistantV2Analytics"
now = datetime.now(timezone.utc).isoformat()

# ---- read mirrored tables via the SQL analytics endpoint ----
# (direct Delta reads fail on mirrored tables that use deletion vectors;
#  synapsesql reads through the SQL endpoint which handles them correctly)
def read_mirror(table):
    last = None
    for fq in [f"{MIRROR_NAME}.{COSMOS_DATABASE}.{table}", f"{MIRROR_NAME}.dbo.{table}"]:
        try:
            df = spark.read.synapsesql(fq)
            print("read", fq)
            return df
        except Exception as e:
            last = e; print("read failed", fq, str(e)[:160])
    raise RuntimeError(f"could not read {table}: {last}")

turns = read_mirror("OptimizationTurns")
print("OptimizationTurns rows:", turns.count())
try:
    trips = read_mirror("Trips")
except Exception as e:
    print("Trips not available:", e); trips = None

# ---- price each turn (join on deployment name) ----
pricing = spark.createDataFrame(
    [("gpt-4.1-mini", 0.40, 1.60), ("gpt-5-nano", 0.05, 0.40), ("gpt-5.1", 1.25, 10.0)],
    ["dep", "pin", "pout"])
priced = (turns.join(pricing, turns.model_deployment == pricing.dep, "left")
          .withColumn("pin", F.coalesce(F.col("pin"), F.lit(0.40)))
          .withColumn("pout", F.coalesce(F.col("pout"), F.lit(1.60)))
          .withColumn("cost", (F.col("input_tokens") * F.col("pin")
                               + F.col("output_tokens") * F.col("pout")) / F.lit(1000000.0)))

# ---- per-tier cost docs ----
tier_costs = (priced.groupBy("tenantId", "model_tier", "model_deployment")
    .agg(F.count(F.lit(1)).alias("turns"),
         F.sum("total_tokens").alias("tokens"),
         F.round(F.sum("cost"), 6).alias("estimated_cost_usd"))
    .withColumn("id", F.concat_ws("::", F.lit("tier"), F.col("tenantId"), F.col("model_tier"), F.col("model_deployment")))
    .withColumn("type", F.lit("tier_cost"))
    .withColumn("computed_at", F.lit(now)))

# ---- per-tenant metrics docs ----
metrics = (priced.groupBy("tenantId")
    .agg(F.count(F.lit(1)).alias("total_turns"),
         F.sum("total_tokens").alias("total_tokens"),
         F.round(F.sum("cost"), 4).alias("estimated_cost_usd"),
         F.sum(F.when(F.col("output_tokens") < 60, 1).otherwise(0)).alias("trivial_turns"),
         F.countDistinct("model_deployment").alias("distinct_models"))
    .withColumn("trivial_pct", F.round(100 * F.col("trivial_turns") / F.col("total_turns"), 1)))

if trips is not None:
    conf = (trips.filter(F.col("status").isin("confirmed", "completed"))
            .groupBy("tenantId").count().withColumnRenamed("count", "confirmed_outcomes"))
    metrics = metrics.join(conf, "tenantId", "left")
else:
    metrics = metrics.withColumn("confirmed_outcomes", F.lit(0))
metrics = (metrics.withColumn("confirmed_outcomes", F.coalesce(F.col("confirmed_outcomes"), F.lit(0)))
    .withColumn("cost_per_outcome_usd", F.when(F.col("confirmed_outcomes") > 0,
                F.round(F.col("estimated_cost_usd") / F.col("confirmed_outcomes"), 4)).otherwise(F.lit(None).cast("double")))
    .withColumn("id", F.concat(F.lit("metrics::"), F.col("tenantId")))
    .withColumn("type", F.lit("metrics"))
    .withColumn("computed_at", F.lit(now)))

print("tenants:", metrics.count(), "tier rows:", tier_costs.count())

# ---- reverse-ETL: write both to Cosmos via the Spark connector (Fabric AAD) ----
cosmos_write = {
    "spark.cosmos.accountEndpoint": COSMOS_ENDPOINT,
    "spark.cosmos.account.tenantId": TENANT_ID,
    "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
    "spark.cosmos.auth.type": "AccessToken",
    "spark.cosmos.useGatewayMode": "true",
    "spark.cosmos.database": COSMOS_DATABASE,
    "spark.cosmos.container": INSIGHTS_CONTAINER,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "spark.cosmos.write.bulk.enabled": "true",
}
metrics.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
tier_costs.write.format("cosmos.oltp").options(**cosmos_write).mode("append").save()
print(f"Reverse-ETL complete -> {COSMOS_DATABASE}/{INSIGHTS_CONTAINER}")
